In [4]:
# ============================================================
# ASSIGNMENT NLP - 5
# Fine-tuning DistilBERT for POS Tagging and Chunking
# Corrected: uses Parquet dataset + processing_class
# ============================================================

!pip install -q transformers datasets evaluate seqeval accelerate

import numpy as np
import torch
from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

from pprint import pprint

# STEP 1: Set seed and device
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# STEP 2: Load dataset safely
dataset = load_dataset(
    "eriktks/conll2003",
    revision="convert/parquet"
)

print("\nDataset loaded successfully:")
print(dataset)

print("\nOne training example:")
pprint(dataset["train"][0])

# STEP 3: Select label columns
pos_label_col = "pos_tags"
chunk_label_col = "chunk_tags"

pos_label_list = dataset["train"].features[pos_label_col].feature.names
chunk_label_list = dataset["train"].features[chunk_label_col].feature.names

print("\nPOS Labels:")
print(pos_label_list)

print("\nChunk Labels:")
print(chunk_label_list)

# STEP 4: Load tokenizer
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print("\nTokenizer loaded successfully")

# STEP 5: Understand tokenization
example = dataset["train"][0]

tokenized_input = tokenizer(
    example["tokens"],
    is_split_into_words=True
)

tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
word_ids = tokenized_input.word_ids()

print("\nOriginal Tokens:")
print(example["tokens"])

print("\nAfter BERT Tokenization:")
print(tokens)

print("\nWord IDs:")
print(word_ids)

print("\nOriginal POS Tags:")
print([pos_label_list[i] for i in example[pos_label_col]])

print("\nOriginal Chunk Tags:")
print([chunk_label_list[i] for i in example[chunk_label_col]])

# STEP 6: Tokenization and label alignment
def tokenize_and_align_labels(examples, label_column):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    aligned_labels = []

    for i, labels in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)

            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

# STEP 7: Use small dataset for fast training
train_data = dataset["train"].select(range(500))
val_data = dataset["validation"].select(range(100))

print("\nTraining samples:", len(train_data))
print("Validation samples:", len(val_data))

# STEP 8: Evaluation metric
seqeval = evaluate.load("seqeval")

def create_compute_metrics(label_list):
    def compute_metrics(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)

        true_predictions = [
            [label_list[p] for p, l in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        true_labels = [
            [label_list[l] for p, l in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        results = seqeval.compute(
            predictions=true_predictions,
            references=true_labels
        )

        return {
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"]
        }

    return compute_metrics

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# STEP 9: Training function
def train_model(task_name, label_column, label_list):
    print("\n======================================")
    print("Training Task:", task_name)
    print("======================================")

    tokenized_train = train_data.map(
        lambda x: tokenize_and_align_labels(x, label_column),
        batched=True
    )

    tokenized_val = val_data.map(
        lambda x: tokenize_and_align_labels(x, label_column),
        batched=True
    )

    id2label = {i: label for i, label in enumerate(label_list)}
    label2id = {label: i for i, label in enumerate(label_list)}

    model = AutoModelForTokenClassification.from_pretrained(
        model_checkpoint,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id
    )

    training_args = TrainingArguments(
        output_dir=f"./{task_name}_model",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=1,
        weight_decay=0.01,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=create_compute_metrics(label_list)
    )

    trainer.train()
    results = trainer.evaluate()

    print("\nEvaluation Result for", task_name)
    print(results)

    return model, trainer

# STEP 10: Train POS model
pos_model, pos_trainer = train_model(
    task_name="POS_Tagging",
    label_column=pos_label_col,
    label_list=pos_label_list
)

# STEP 11: Train Chunking model
chunk_model, chunk_trainer = train_model(
    task_name="Chunking",
    label_column=chunk_label_col,
    label_list=chunk_label_list
)

# STEP 12: Prediction function
def predict_tags(sentence, model, label_list):
    words = sentence.split()

    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)[0].tolist()
    word_ids = inputs.word_ids()

    final_result = []
    previous_word_id = None

    for index, word_id in enumerate(word_ids):
        if word_id is None:
            continue

        if word_id != previous_word_id:
            word = words[word_id]
            tag = label_list[predictions[index]]
            final_result.append((word, tag))

        previous_word_id = word_id

    return final_result

# STEP 13: Test custom sentence
sentence = "John works at Google in California"

print("\nCustom Sentence:")
print(sentence)

print("\nPOS Tagging Output:")
pprint(predict_tags(sentence, pos_model, pos_label_list))

print("\nChunking Output:")
pprint(predict_tags(sentence, chunk_model, chunk_label_list))


# STEP 14: Save models
pos_model.save_pretrained("./final_pos_model")
chunk_model.save_pretrained("./final_chunk_model")
tokenizer.save_pretrained("./final_tokenizer")

print("\nAll steps completed successfully.")
print("Now save notebook as .ipynb and upload it to GitHub.")

Device: cpu

Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

One training example:
{'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'id': '0',
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'tokens': ['EU',
            'rejects',
            'German',
            'call',
            'to',
            'boycott',
            'British',
            'lamb',
            '.']}

POS Labels:
['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,2.627220,0.145390,0.036607,0.058488,0.283704


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,1.407100,0.295110,0.255848,0.274080,0.485926



Evaluation Result for Chunking
{'eval_loss': 1.407099723815918, 'eval_precision': 0.2951096121416526, 'eval_recall': 0.25584795321637427, 'eval_f1': 0.274079874706343, 'eval_accuracy': 0.48592592592592593, 'eval_runtime': 4.5314, 'eval_samples_per_second': 22.068, 'eval_steps_per_second': 2.869, 'epoch': 1.0}

Custom Sentence:
John works at Google in California

POS Tagging Output:
[('John', 'NNP'),
 ('works', 'NNP'),
 ('at', 'NNP'),
 ('Google', 'NNP'),
 ('in', 'NNP'),
 ('California', 'NNP')]

Chunking Output:
[('John', 'I-NP'),
 ('works', 'I-NP'),
 ('at', 'I-NP'),
 ('Google', 'I-NP'),
 ('in', 'I-NP'),
 ('California', 'I-NP')]

REPORT / BLOG

Dataset Used:
CoNLL-2003 dataset loaded from Parquet format.

Model Used:
DistilBERT using AutoModelForTokenClassification.

POS Tagging:
POS tagging means assigning grammar labels to each word.
Example: noun, verb, adjective, preposition.

Chunking:
Chunking means grouping words into meaningful phrases.
Example: noun phrase, verb phrase, preposi

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


All steps completed successfully.
Now save notebook as .ipynb and upload it to GitHub.
